# Process Monitor Analysis Notebook

This notebook connects to the database and provides tools to analyze process monitoring data from the `process_monitor_logs` table. You can:

1. View the most recent run(s)
2. Analyze the stages of a specific run
3. Compare performance across multiple runs
4. Examine specific stages (like database queries) in detail

In [ ]:
import os
import json
import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import psycopg2
from psycopg2.extras import RealDictCursor
from IPython.display import display, HTML

# Add iris to Python path if needed
import sys
if '..' not in sys.path:
    sys.path.append('..')

# Import database connection function from iris
from iris.src.initial_setup.db_config import connect_to_db

# Set environment (this was previously imported from db_config)
ENVIRONMENT = "local"  # Change this to "rbc" if needed

## Database Connection

Let's set up and test the database connection.

In [ ]:
# Test connection
try:
    conn = connect_to_db(ENVIRONMENT)
    if conn:
        print(f"✅ Connected to database successfully (Environment: {ENVIRONMENT})")
        conn.close()
    else:
        print(f"❌ Failed to connect to database (Environment: {ENVIRONMENT})")
except Exception as e:
    print(f"❌ Error connecting to database: {e}")

## Query Functions

These functions help retrieve and analyze process monitoring data.

In [ ]:
def get_recent_runs(limit=1):
    """Get the most recent run UUIDs from the process_monitor_logs table.
    
    Args:
        limit (int): Number of recent runs to retrieve
        
    Returns:
        pandas.DataFrame: DataFrame with run information
    """
    conn = connect_to_db(ENVIRONMENT)
    if not conn:
        print("Error: Failed to get DB connection in get_recent_runs")
        return pd.DataFrame() # Return empty DataFrame on connection failure
    try:
        with conn.cursor(cursor_factory=RealDictCursor) as cur:
            query = """
            WITH run_summary AS (
                SELECT 
                    run_uuid,
                    MIN(stage_start_time) as run_start_time,
                    MAX(stage_end_time) as run_end_time,
                    COUNT(*) as total_stages,
                    SUM(COALESCE(total_tokens, 0)) as total_tokens,
                    SUM(COALESCE(total_cost, 0)) as total_cost
                FROM process_monitor_logs
                GROUP BY run_uuid
            )
            SELECT 
                run_uuid,
                run_start_time,
                run_end_time,
                EXTRACT(EPOCH FROM (run_end_time - run_start_time)) as duration_seconds,
                total_stages,
                total_tokens,
                total_cost
            FROM run_summary
            ORDER BY run_start_time DESC
            LIMIT %s
            """
            cur.execute(query, (limit,))
            result = cur.fetchall()
            
            # Convert to DataFrame
            if result:
                df = pd.DataFrame(result)
                # Format duration and cost
                if 'duration_seconds' in df.columns:
                    df['duration'] = df['duration_seconds'].apply(lambda x: f"{x:.2f} sec")
                if 'total_cost' in df.columns:
                    df['total_cost'] = df['total_cost'].apply(lambda x: f"${x:.6f}")
                return df
            else:
                print("No runs found in the process_monitor_logs table.")
                return pd.DataFrame()
    finally:
        if conn:
             conn.close()

def get_run_details(run_uuid):
    """Get detailed information about a specific run.
    
    Args:
        run_uuid (str): UUID of the run to analyze
        
    Returns:
        pandas.DataFrame: DataFrame with stage details
    """
    conn = connect_to_db(ENVIRONMENT)
    if not conn:
        print(f"Error: Failed to get DB connection in get_run_details for {run_uuid}")
        return pd.DataFrame()
    try:
        with conn.cursor(cursor_factory=RealDictCursor) as cur:
            query = """
            SELECT 
                log_id,
                stage_name,
                stage_start_time,
                stage_end_time,
                duration_ms,
                total_tokens,
                total_cost,
                status,
                decision_details,
                error_message,
                llm_calls
            FROM process_monitor_logs
            WHERE run_uuid = %s
            ORDER BY stage_start_time
            """
            cur.execute(query, (run_uuid,))
            result = cur.fetchall()
            
            # Convert to DataFrame
            if result:
                df = pd.DataFrame(result)
                # Format durations and costs with better handling for zeros and nulls
                if 'duration_ms' in df.columns:
                    df['duration'] = df['duration_ms'].apply(lambda x: f"{x/1000:.3f} sec" if (x is not None and x > 0) else "0.000 sec" if x == 0 else "N/A")
                if 'total_cost' in df.columns:
                    df['total_cost'] = df['total_cost'].apply(lambda x: f"${x:.6f}" if x is not None else "$0.00")
                if 'total_tokens' in df.columns:
                    df['total_tokens'] = df['total_tokens'].fillna(0).astype(int)
                return df
            else:
                print(f"No stages found for run UUID: {run_uuid}")
                return pd.DataFrame()
    finally:
        if conn:
            conn.close()

def get_multiple_runs(limit=5):
    """Get multiple recent runs for comparison.
    
    Args:
        limit (int): Number of recent runs to retrieve
        
    Returns:
        pandas.DataFrame: DataFrame with run information
    """
    return get_recent_runs(limit)

def get_stage_details(run_uuid, stage_name):
    """Get detailed information about a specific stage in a run.
    
    Args:
        run_uuid (str): UUID of the run
        stage_name (str): Name of the stage to analyze
        
    Returns:
        dict: Stage details including LLM calls
    """
    conn = connect_to_db(ENVIRONMENT)
    if not conn:
        print(f"Error: Failed to get DB connection in get_stage_details for {run_uuid}/{stage_name}")
        return None # Return None on connection failure
    try:
        with conn.cursor(cursor_factory=RealDictCursor) as cur:
            query = """
            SELECT *
            FROM process_monitor_logs
            WHERE run_uuid = %s AND stage_name = %s
            """
            cur.execute(query, (run_uuid, stage_name))
            result = cur.fetchone()
            return result
    finally:
        if conn:
            conn.close()

## Get Most Recent Run

Retrieve and display the most recent process monitoring run.

In [ ]:
# Get the most recent run (change limit to see more)
recent_runs = get_recent_runs(limit=1)

if not recent_runs.empty:
    display(recent_runs)
    
    # Store the most recent run UUID for further analysis
    most_recent_run_uuid = recent_runs['run_uuid'].iloc[0]
    print(f"Most recent run UUID: {most_recent_run_uuid}")
else:
    print("No runs found. Try running a query with the model first.")

## Get Multiple Recent Runs

Retrieve and compare multiple recent runs.

In [ ]:
# Get the 5 most recent runs (adjust the limit as needed)
multiple_runs = get_multiple_runs(limit=5)

if not multiple_runs.empty:
    display(multiple_runs)
else:
    print("No runs found. Try running a query with the model first.")

## Analyze Specific Run

Analyze the stages of a specific run. By default, it uses the most recent run, but you can specify any run UUID.

In [ ]:
# Get a specific run UUID (either by setting it manually or using the most recent one)
try:
    run_uuid_to_analyze = most_recent_run_uuid  # From previous cell
except NameError:
    # If most_recent_run_uuid is not defined, get it now
    recent_runs = get_recent_runs(limit=1)
    if not recent_runs.empty:
        run_uuid_to_analyze = recent_runs['run_uuid'].iloc[0]
    else:
        run_uuid_to_analyze = None
        print("No runs found. Try running a query with the model first.")

# You can also manually set a run UUID here
# run_uuid_to_analyze = "00000000-0000-0000-0000-000000000000"

if run_uuid_to_analyze:
    print(f"Analyzing run UUID: {run_uuid_to_analyze}")
    run_details = get_run_details(run_uuid_to_analyze)
    
    if not run_details.empty:
        # Select and display the most relevant columns
        display_columns = ['stage_name', 'duration', 'total_tokens', 'total_cost', 'status', 'decision_details']
        display(run_details[display_columns])
    else:
        print(f"No stages found for run UUID: {run_uuid_to_analyze}")

## Visualize Stage Durations

Create a bar chart of stage durations for the selected run.

In [ ]:
if 'run_details' in locals() and not run_details.empty:
    # Create a duration bar chart
    plt.figure(figsize=(12, 6))
    
    # Sort by duration for better visualization
    plot_data = run_details.sort_values('duration_ms', ascending=False)
    
    # Plot the durations
    ax = sns.barplot(x='stage_name', y='duration_ms', data=plot_data)
    plt.title('Stage Durations (ms)')
    plt.xlabel('Stage')
    plt.ylabel('Duration (ms)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    
    # Add duration values on top of bars
    for i, bar in enumerate(ax.patches):
        # Only add text if the height is greater than 0
        if bar.get_height() > 0:
            ax.text(
                bar.get_x() + bar.get_width()/2.,
                bar.get_height() + 5,
                f"{bar.get_height():.0f} ms",
                ha='center', va='bottom', rotation=0, size=8
            )
    
    plt.show()
    
    # Create a token usage bar chart (if total_tokens column exists)
    if 'total_tokens' in run_details.columns:
        # Filter out stages with no token usage
        token_data = run_details[run_details['total_tokens'].notna() & (run_details['total_tokens'] > 0)]
        
        if not token_data.empty:
            plt.figure(figsize=(12, 6))
            ax = sns.barplot(x='stage_name', y='total_tokens', data=token_data)
            plt.title('Token Usage by Stage')
            plt.xlabel('Stage')
            plt.ylabel('Total Tokens')
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            
            # Add token values on top of bars
            for i, bar in enumerate(ax.patches):
                if bar.get_height() > 0:
                    ax.text(
                        bar.get_x() + bar.get_width()/2.,
                        bar.get_height() + 5,
                        f"{bar.get_height():.0f}",
                        ha='center', va='bottom', rotation=0, size=8
                    )
            
            plt.show()

## Analyze Database Query Stages

Focus on database query stages (those starting with "db_query_") and display their details.

In [ ]:
if 'run_details' in locals() and not run_details.empty:
    # Filter for database query stages
    db_query_stages = run_details[run_details['stage_name'].str.startswith('db_query_')]
    
    if not db_query_stages.empty:
        print(f"Found {len(db_query_stages)} database query stages")
        display_columns = ['stage_name', 'duration', 'total_tokens', 'total_cost', 'status', 'decision_details']
        display(db_query_stages[display_columns])
        
        # Plot database query durations
        plt.figure(figsize=(12, 6))
        plot_data = db_query_stages.sort_values('duration_ms', ascending=False)
        
        # Extract the database name from the stage_name
        plot_data['database'] = plot_data['stage_name'].apply(
            lambda x: '_'.join(x.split('_')[2:-1]) if len(x.split('_')) > 3 else x
        )
        
        ax = sns.barplot(x='database', y='duration_ms', data=plot_data)
        plt.title('Database Query Durations (ms)')
        plt.xlabel('Database')
        plt.ylabel('Duration (ms)')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        
        # Add duration values on top of bars
        for i, bar in enumerate(ax.patches):
            if bar.get_height() > 0:
                ax.text(
                    bar.get_x() + bar.get_width()/2.,
                    bar.get_height() + 5,
                    f"{bar.get_height():.0f} ms",
                    ha='center', va='bottom', rotation=0, size=8
                )
        
        plt.show()
    else:
        print("No database query stages found in this run.")

## Examine LLM Calls Details

Analyze the detailed LLM call information for a specific stage.

In [ ]:
# Select a stage to examine in detail
if 'run_details' in locals() and not run_details.empty:
    # Display stage names for selection
    print("Available stages:")
    for i, stage_name in enumerate(run_details['stage_name']):
        print(f"{i+1}. {stage_name}")
    
    # Select a stage (you can change this index)
    selected_stage_index = 0  # First stage by default
    
    if len(run_details) > 0:
        if selected_stage_index < len(run_details):
            selected_stage = run_details['stage_name'].iloc[selected_stage_index]
            print(f"\nSelected stage: {selected_stage}")
            
            # Get detailed information for the selected stage
            stage_info = get_stage_details(run_uuid_to_analyze, selected_stage)
            
            if stage_info:
                # Display basic stage information
                basic_info = {
                    'Stage Name': stage_info['stage_name'],
                    'Start Time': stage_info['stage_start_time'],
                    'End Time': stage_info['stage_end_time'],
                    'Duration (ms)': stage_info['duration_ms'],
                    'Status': stage_info['status'],
                    'Total Tokens': stage_info['total_tokens'],
                    'Total Cost': f"${stage_info['total_cost']:.6f}" if stage_info['total_cost'] else "$0.00",
                    'Decision Details': stage_info['decision_details'],
                    'Error Message': stage_info['error_message'] or "None"
                }
                display(pd.DataFrame([basic_info]).T)
                
                # Examine LLM calls if available
                if stage_info['llm_calls']:
                    try:
                        llm_calls = json.loads(stage_info['llm_calls']) if isinstance(stage_info['llm_calls'], str) else stage_info['llm_calls']
                        if llm_calls:
                            print("\nLLM Calls:")
                            llm_calls_df = pd.DataFrame(llm_calls)
                            display(llm_calls_df)
                        else:
                            print("\nNo LLM calls recorded for this stage.")
                    except (ValueError, TypeError) as e:
                        print(f"\nError parsing LLM calls: {e}")
                        print(f"Raw LLM calls data: {stage_info['llm_calls']}")
                else:
                    print("\nNo LLM calls recorded for this stage.")
            else:
                print(f"Could not retrieve details for stage: {selected_stage}")
        else:
            print("Invalid stage index")
    else:
        print("No stages found in the run details")

## Custom Query

You can also write your own custom SQL query to analyze the process monitoring data.

In [ ]:
def run_custom_query(query, params=None):
    """Run a custom SQL query against the process_monitor_logs table.
    
    Args:
        query (str): SQL query to run
        params (tuple, optional): Parameters for the query
        
    Returns:
        pandas.DataFrame: Query results
    """
    conn = connect_to_db(ENVIRONMENT)
    if not conn:
        print("Error: Failed to get DB connection in run_custom_query")
        return pd.DataFrame()
    try:
        with conn.cursor(cursor_factory=RealDictCursor) as cur:
            cur.execute(query, params or ())
            result = cur.fetchall()
            return pd.DataFrame(result)
    finally:
        if conn:
            conn.close()

# Example: Get average duration by stage type across all runs
custom_query = """
SELECT 
    stage_name,
    COUNT(*) as num_occurrences,
    AVG(duration_ms) as avg_duration_ms,
    AVG(total_tokens) as avg_tokens,
    AVG(total_cost) as avg_cost
FROM process_monitor_logs
GROUP BY stage_name
ORDER BY avg_duration_ms DESC
LIMIT 10
"""

# Run the custom query
try:
    custom_results = run_custom_query(custom_query)
    if not custom_results.empty:
        display(custom_results)
    else:
        print("No results returned from custom query.")
except Exception as e:
    print(f"Error running custom query: {e}")

In [ ]:
# Example: Get total token usage and cost by run_uuid, with timing information
token_query = """
WITH run_summary AS (
    SELECT 
        run_uuid,
        MIN(stage_start_time) as run_start_time,
        SUM(COALESCE(total_tokens, 0)) as total_tokens,
        SUM(COALESCE(total_cost, 0)) as total_cost
    FROM process_monitor_logs
    GROUP BY run_uuid
)
SELECT 
    run_uuid,
    run_start_time,
    total_tokens,
    total_cost
FROM run_summary
ORDER BY run_start_time DESC
LIMIT 10
"""

# Run the query
try:
    token_results = run_custom_query(token_query)
    if not token_results.empty:
        # Format cost as currency
        token_results['formatted_cost'] = token_results['total_cost'].apply(lambda x: f"${x:.6f}" if x is not None else "$0.00")
        display(token_results)
        
        # Plot token usage over time
        if len(token_results) > 0:
            plt.figure(figsize=(12, 6))
            plt.bar(token_results['run_start_time'], token_results['total_tokens'])
            plt.title('Token Usage by Run')
            plt.xlabel('Run Start Time')
            plt.ylabel('Total Tokens')
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()
    else:
        print("No token usage data found.")
except Exception as e:
    print(f"Error running token query: {e}")